In [2452]:
import re
import unicodedata
import pandas as pd

In [2453]:
df = pd.read_csv(r"dataset\filtering\news_balanced.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
''')
df.head()


      Shape: (100, 6)
      Columns: ['content_id', 'text', 'section', 'published_date', 'category', 'keywords']



,content_id,text,section,published_date,category,keywords
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29 00:00:00,5G,"5G,Internet,Technology,AI,Smart cities"
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17 00:00:00,Gadgets,"Gadgets,Technology"
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04 00:00:00,Gadgets,"Gadgets,Courts Crime"
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30 00:00:00,Gadgets,"Gadgets,Technology"
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13 00:00:00,Gadgets,"Gadgets,Environment"


In [2454]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"''|``|‘’", '"', text)
    
    text = text.replace('\xad', '')
    
    text = re.sub(r'\b(?:https?://|www\.)?\S+\.\S+(?:/\S*)?', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    
    # replace control chars with space instead of deleting
    text = ''.join(ch if unicodedata.category(ch)[0] != "C" else ' ' for ch in text)
    
    # now fix glued sentences
    text = re.sub(r'([.!?])([A-Z])', r'\1 \2', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    text = re.sub(r'©.*?(?:\.|$)', '', text)
    
    return text.strip()

In [2455]:
df['clean_text'] = df['text'].apply(clean_text)

In [2456]:
df['published_date'] = pd.to_datetime(df['published_date'])

In [2457]:
df.text[3]

'I knew my efforts to learn Japanese before my trip to Japan last fall would be no match for a fire ritual at a Buddhist temple in Tokyo, so at the beginning of the ceremony, I discreetly slipped in one earbud.\r\n\r\nTwo monks pounded giant taiko drums as others chanted in unison under a cloud of fragrant smoke. Then the priest delivered closing remarks in Japanese. And in a miracle of technology, I understood quite a bit of it using Apple’s new Live Translation feature.\r\n\r\nAs everyone filed out, I repeated, in English, some of the priest’s comments to my guide, Keiko Hatada, who taught English for 30 years and has led custom tours of Tokyo for the past decade. I wanted to make sure I had understood things correctly.\r\n\r\nI recounted the priest’s admonition to set aside unwholesome feelings of anger and greed, and work instead to show compassion and generosity, as well as his reminder that his temple was still accepting donations for those affected by the 2011 earthquake and tsu

In [2458]:
clean_text(df.clean_text[3])

'I knew my efforts to learn Japanese before my trip to Japan last fall would be no match for a fire ritual at a Buddhist temple in Tokyo, so at the beginning of the ceremony, I discreetly slipped in one earbud. Two monks pounded giant taiko drums as others chanted in unison under a cloud of fragrant smoke. Then the priest delivered closing remarks in Japanese. And in a miracle of technology, I understood quite a bit of it using Apple’s new Live Translation feature. As everyone filed out, I repeated, in English, some of the priest’s comments to my guide, Keiko Hatada, who taught English for 30 years and has led custom tours of Tokyo for the past decade. I wanted to make sure I had understood things correctly. I recounted the priest’s admonition to set aside unwholesome feelings of anger and greed, and work instead to show compassion and generosity, as well as his reminder that his temple was still accepting donations for those affected by the 2011 earthquake and tsunami. “You told me yo

In [2459]:
df.head()

,content_id,text,section,published_date,category,keywords,clean_text
0,1718879,With artificial intelligence (AI) in the mix a...,Tech,2025-09-29,5G,"5G,Internet,Technology,AI,Smart cities",With artificial intelligence (AI) in the mix a...
1,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17,Gadgets,"Gadgets,Technology","CUPERTINO: Until now, the AirPods Pro were all..."
2,1297028,"WATERTOWN, New York: A Watertown man was arres...",Tech,2024-03-04,Gadgets,"Gadgets,Courts Crime","WATERTOWN, New York: A Watertown man was arres..."
3,1782746,I knew my efforts to learn Japanese before my ...,Tech,2025-12-30,Gadgets,"Gadgets,Technology",I knew my efforts to learn Japanese before my ...
4,1345607,LONDON: British fire chiefs and recycling camp...,Tech,2024-05-13,Gadgets,"Gadgets,Environment",LONDON: British fire chiefs and recycling camp...


In [2460]:
df['timestamp'] = df['published_date'].astype('int64') // 10**9

In [2461]:
df['year'] = df['published_date'].dt.year
df['month'] = df['published_date'].dt.month
df['week'] = df['published_date'].dt.isocalendar().week

In [2462]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2463]:
import re

def merge_quote_splits(sentences):
    merged = []
    buffer = ""

    for sent in sentences:
        if buffer:
            buffer += " " + sent
            if buffer.count('"') % 2 == 0:
                merged.append(buffer.strip())
                buffer = ""
        else:
            if sent.count('"') % 2 != 0:
                buffer = sent
            else:
                merged.append(sent)

    if buffer:
        merged.append(buffer)

    return merged

In [2464]:
def secondary_split(sent):
    return re.split(r'(?<!")\.\s+(?=[A-Z])', sent)

def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [2465]:
def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [2466]:
df['sentences'] = df['clean_text'].apply(
    lambda x: refine_sentences(
        merge_quote_splits(sent_tokenize(x))
    )
)

In [2467]:
df_sent = df.explode('sentences').reset_index(drop=True)
df_sent.rename(columns={'sentences': 'sentence'}, inplace=True)

In [2468]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(2975, 12)


AI         1391
Gadgets    1156
5G          428
Name: category, dtype: int64

In [2469]:
df_sent = df_sent[df_sent['sentence'].str.len() > 40]

In [2470]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(2819, 12)


AI         1320
Gadgets    1091
5G          408
Name: category, dtype: int64

In [ ]:
boilerplate_pattern= r'[.|"]\s*[–-]\s*[A-Za-z\s]{1,40}$'

df_sent['sentence'] = df_sent['sentence'].str.replace(
    boilerplate_pattern,
    '',
    regex=True
)

In [2472]:
df_final = df_sent[['content_id', 'sentence', 'published_date', 'timestamp', 'category']]

print(df_final.shape)
df_final.head()

(2819, 5)


,content_id,sentence,published_date,timestamp,category
0,1718879,With artificial intelligence (AI) in the mix a...,2025-09-29,1759104000,5G
1,1718879,"Take Putrajaya Corporation, for instance, whic...",2025-09-29,1759104000,5G
2,1718879,"At its SCEKL booth, the city featured everythi...",2025-09-29,1759104000,5G
3,1718879,"In the case of smart traffic lights, AI stream...",2025-09-29,1759104000,5G
4,1718879,The system achieves this by having CCTV camera...,2025-09-29,1759104000,5G


In [2473]:
df_final.to_csv(r"dataset\preprocessing\prePro-news_balanced.csv", index=False)